In [ ]:
## https://colab.research.google.com/drive/1g_fS5SORcOlm91hJbjwEgox3Uv9f6pMC?usp=sharing

In [ ]:
from datasets import load_dataset
ds = load_dataset('merve/vqav2-small', trust_remote_code=True) # the provided hugging face dataset for inference
ds # check the schema

In [ ]:
## checking and random sample
import random
def create_random():
  rand_index = random.randint(0,len(ds["validation"])-1)
  return rand_index

import PIL
def half_image(image:PIL.Image) -> PIL.Image:
  image = image.resize((image.size[0]//2,image.size[1]//2))
  return image

import time
from IPython.display import display  , clear_output

for i in range(10):
  clear_output(wait=True) # to clear output scrren
  random_sample = ds["validation"][create_random()]
  print(f"[INFO] Index : {i}")
  display(half_image(random_sample["image"]))
  print(f'[INFO] Question:\n{random_sample["question"]}')
  print(f'[INFO] Answer:\n{random_sample["multiple_choice_answer"]}')
  time.sleep(5)

In [ ]:
# ## creating collate function to see the format of dataset

# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "text", "text": "Answer briefly."},
#             {"type": "image"},  # Placeholder for the image
#             {"type": "text", "text": "What is the primary object in this image?"}
#         ]
#     },
#     {
#         "role": "assistant",
#         "content": [
#             {"type": "text", "text": "A red car."}
#         ]
#     }
# ]

## requried format from dataset

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Idefics3ForConditionalGeneration

USE_LORA = False
USE_QLORA = True
SMOL = True

model_id = "HuggingFaceTB/SmolVLM-Base" if SMOL else "HuggingFaceM4/Idefics3-8B-Llama3"
## this inference is just to look the special tokens for image
processor = AutoProcessor.from_pretrained(
    model_id
)

In [ ]:
random_sample

In [ ]:
question = random_sample["question"]
image = random_sample["image"]
answer = random_sample["multiple_choice_answer"] # breakdown fro fast inference
message = [
    {
        "role":"user", # what user will send
        "content":[
            {"type":"text","text":"Answer briefly."}, # here the system prompt from user
            {"type":"image"}, # this will replace with image token by processor
            {"type":"text","text":question} # here the question give
        ]
    },
    {
        "role":"assistant",
        "content":[
            {"type":"text","text":answer} # the answer for supervisied apprach
        ]
    }
]

In [ ]:
text = processor.apply_chat_template(message,add_generation=True,tokenize=False) # converted to chat template
text # viewing
## this is that the "image" is replaced with its token <image> and other essential tokens also added

In [ ]:
text.strip() # so this is done to remove the extra warning

In [ ]:
## creating two list to simulate batch inference from processor
images = []
texts = []
## adding values
images.append([image])
texts.append(text.strip())

In [ ]:
## batch inference simulated

batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

In [ ]:
batch.keys()
## return : input_ids(labels) , attention_mask , pixel_values , pixel_masks

In [ ]:
labels = batch["input_ids"].clone()
labels[labels == processor.tokenizer.pad_token_id] = -100

In [ ]:
import numpy as np
np.where(labels == -100)

In [ ]:
labels = batch["input_ids"].clone() # copy of inpu_ids taken
labels[labels == processor.tokenizer.pad_token_id] = -100
labels[labels == image_token_id] = -100
batch["labels"] = labels

In [ ]:
## step 1 : preprocessing done !!

## input_ids ;= are the tokens converted from the formated text and we will strip it to create the label i.e displayinh output